<a href="https://colab.research.google.com/github/doorandy/pix2pix/blob/master/Preprocesamiento_imagenes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
!pip install ultralytics

!pip install roboflow

!pip install rawpy


"""
Script de preprocesamiento, entrenamiento e inferencia masiva local en Colab.
"""

import os
import psutil
import glob
import torch
import rawpy
import cv2
import zipfile
import shutil
import time
import random
import yaml
from tqdm import tqdm
from ultralytics import YOLO
from roboflow import Roboflow
from google.colab import userdata

# Instalación de dependencias
!pip install ultralytics roboflow rawpy

# --- 1. Descarga y preparación del dataset ---
if not os.path.exists("dataset_agua"):
    !wget "https://zenodo.org/records/14219501/files/Fotos%20NTU.zip?download=1" -O dataset.zip
    with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
        zip_ref.extractall("dataset_agua")

# Creación de subconjunto para etiquetado
input_path = 'dataset_agua/Fotos NTU'
output_train = 'muestras_para_etiquetar'

if not os.path.exists(output_train):
    os.makedirs(output_train)

all_nef = [f for f in os.listdir(input_path) if f.lower().endswith('.nef')]
sample_nef = random.sample(all_nef, 40)

for filename in tqdm(sample_nef):
    with rawpy.imread(os.path.join(input_path, filename)) as raw:
        img = raw.postprocess(use_camera_wb=True)
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(output_train, filename.replace('.NEF', '.png')), img_bgr)

# --- 2. Integración con Roboflow y entrenamiento ---
api_key_oculta = userdata.get('ROBOFLOW_KEY')
rf = Roboflow(api_key=api_key_oculta)

project = rf.workspace("andy-mta").project("detectar_frascos")
version = project.version(2)
dataset = version.download("yolov8")

# Corrección de rutas en data.yaml para entorno Colab
base_path = '/content/detectar_frascos-2'
yaml_file = os.path.join(base_path, 'data.yaml')

if os.path.exists(os.path.join(base_path, 'test')) and not os.path.exists(os.path.join(base_path, 'valid')):
    shutil.copytree(os.path.join(base_path, 'test'), os.path.join(base_path, 'valid'))

with open(yaml_file, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(base_path, 'train', 'images')
data['val'] = os.path.join(base_path, 'valid', 'images')
if 'test' in data:
    data['test'] = os.path.join(base_path, 'test', 'images')

with open(yaml_file, 'w') as f:
    yaml.dump(data, f)

# Entrenamiento del modelo
model = YOLO('yolov8n.pt')
model.train(
    data=os.path.join(base_path, 'data.yaml'),
    epochs=50,
    imgsz=640,
    device=0,
    plots=True,
    project='/content/runs',
    name='train'
)

# --- 3. Procesamiento masivo y evaluación de recursos ---
def get_ram_usage():
    """Retorna el uso de memoria RSS actual en MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

ram_base = get_ram_usage()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# Carga del modelo entrenado
model_candidates = glob.glob('/content/runs/train*/weights/best.pt')
model_candidates.sort(key=os.path.getmtime)
model_path = model_candidates[-1]
model_tesis = YOLO(model_path)
ram_con_modelo = get_ram_usage()

output_path = '/content/DATASET_FINAL_RECORTADO'
if not os.path.exists(output_path):
    os.makedirs(output_path)

nef_files = [f for f in os.listdir(input_path) if f.lower().endswith('.nef')]
start_time = time.time()

for filename in tqdm(nef_files):
    try:
        with rawpy.imread(os.path.join(input_path, filename)) as raw:
            img_rgb = raw.postprocess(use_camera_wb=True)
            img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

        results = model_tesis(img_bgr, verbose=False)

        if len(results[0].boxes) > 0:
            box = results[0].boxes[0].xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, box)
            crop = img_bgr[max(0, y1):min(img_bgr.shape[0], y2),
                           max(0, x1):min(img_bgr.shape[1], x2)]
            cv2.imwrite(os.path.join(output_path, filename.lower().replace('.nef', '.png')), crop)
    except Exception:
        continue

end_time = time.time()

# --- 4. Reporte y exportación ---
ram_final = get_ram_usage()
vram_peak = torch.cuda.max_memory_allocated() / (1024 * 1024) if torch.cuda.is_available() else 0.0
total_duration = (end_time - start_time) / 60

report_path = "/content/reporte_memoria_script.txt"
with open(report_path, "w") as f:
    f.write("REPORTE DE USO DE RECURSOS COMPUTACIONALES\n")
    f.write("--------------------------------------------------\n")
    f.write(f"Tiempo total de procesamiento: {total_duration:.2f} minutos\n")
    f.write(f"Archivos procesados: {len(nef_files)}\n\n")
    f.write("Estadísticas de memoria RAM:\n")
    f.write(f"- RAM carga modelo: {ram_con_modelo - ram_base:.2f} MB\n")
    f.write(f"- Pico RAM: {ram_final:.2f} MB\n")
    f.write(f"- Impacto neto: {ram_final - ram_base:.2f} MB\n\n")
    f.write("Estadísticas de memoria de video:\n")
    f.write(f"- Pico asignación VRAM: {vram_peak:.2f} MB\n")

archive_name = 'Dataset_Recortado_Turbidez_Final'
if os.path.exists(output_path):
    shutil.make_archive(archive_name, 'zip', output_path)

Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/detectar_frascos-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience

 89%|████████▉ | 624/700 [39:03<04:42,  3.71s/it]